# Texte LONG que le modele maitrise — extraction complete

Ce notebook demontre le modele sur un **texte long ecrit dans le style du corpus d'entrainement**.
Resultat teste : **18 entites extraites, 16 types de traits differents, toutes correctes.**

## Pourquoi celui-ci marche (et pas n'importe quel texte)
Le modele a ete entraine sur de la **prose scientifique qui nomme explicitement les traits DEB**.
Il reconnait tres bien des formulations comme :

| Formulation (style corpus) | Trait |
|---|---|
| « at a stable water temperature of **17 ° C** » | `T` |
| « the **age at birth** was recorded as **12 days** » | `ab` |
| « the initial wet weight of the freshly laid egg ... **0.85 grams** » | `Ww0` |
| « a total **length at birth** of **0.6 cm** » | `Lb` |
| « the **wet weight at birth** is approximately **1.2 grams** » | `Wwb` |
| « the **time since birth at weaning** is **21 days** » | `tx` |
| « a **wet weight at weaning** of **3.4 grams** » | `Wwx` |
| « the total **length at puberty** is **8.2 cm** » | `Lp` |
| « a **wet weight at puberty** of **24 grams** » | `Wwp` |
| « attain **puberty** at approximately **730 days after hatching** » | `tp` |
| « the **time since birth at first brood** ... **950 days** » | `tR` |
| « attains a **maximum recorded total length** of **21.5 cm** » | `Li` |
| « the **ultimate wet weight** ... estimated at **145 grams** » | `Wwi` |
| « can produce approximately **320 eggs per day** » | `Ri` |
| « the **gonadosomatic index ( GSI )** was recorded at **18 %** » | `GSI` |
| « the maximum documented **life span** ... is **8 years** » | `am` |

A l'inverse, une formulation « grand public » (« newborn kits weigh 45 g », « survive beyond 9 years »)
est beaucoup moins bien reconnue : le modele est **specialise sur le vocabulaire DEB**.

In [1]:
import html as _html
import re
from pathlib import Path

import pandas as pd
import torch
from IPython.display import HTML, display
from transformers import AutoModelForTokenClassification, AutoTokenizer

ROOT = Path.cwd()
while not (ROOT / "outputs").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MODEL_DIR = ROOT / "outputs" / "bio" / "bert" / "baseline" / "final_model"

device = "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(str(MODEL_DIR)).eval().to(device)
id2label = {int(k): v for k, v in model.config.id2label.items()}
MAX_LEN = min(tokenizer.model_max_length, model.config.max_position_embeddings)

_SENT = re.compile(r"(?<=[.!?])\s+")
_WORD = re.compile(r"\d+(?:[.,]\d+)*|\w+|[^\w\s]")


def _spans(tags, offsets):
    spans, cur = [], None
    for tag, (start, end) in zip(tags, offsets):
        if tag == "O":
            if cur: spans.append(cur); cur = None
            continue
        prefix, etype = tag.split("-", 1)
        if prefix == "B":
            if cur: spans.append(cur)
            cur = {"type": etype, "start": start, "end": end}
        elif cur and cur["type"] == etype:
            cur["end"] = end
    if cur: spans.append(cur)
    return spans


def analyse(text):
    out = []
    for sentence in [s for s in _SENT.split(text.strip()) if s]:
        pairs = [(m.group(), (m.start(), m.end())) for m in _WORD.finditer(sentence)]
        if not pairs:
            out.append((sentence, [])); continue
        words, offsets = zip(*pairs)
        enc = tokenizer(list(words), is_split_into_words=True, return_tensors="pt",
                        truncation=True, max_length=MAX_LEN)
        word_ids = enc.word_ids(batch_index=0)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            pred = model(**enc).logits[0].argmax(-1).tolist()
        seen, tags, offs = set(), [], []
        for i, w in enumerate(word_ids):
            if w is None or w in seen: continue
            seen.add(w); tags.append(id2label[pred[i]]); offs.append(offsets[w])
        out.append((sentence, _spans(tags, offs)))
    return out


def table(text):
    return pd.DataFrame([{"Type": sp["type"], "Texte extrait": sent[sp["start"]:sp["end"]]}
                         for sent, spans in analyse(text) for sp in spans])


_PALETTE = ["#c6dbef", "#c7e9c0", "#fdd0a2", "#dadaeb", "#fdd49e", "#d9f0d3",
            "#fde0dd", "#e5f5e0", "#bcbddc", "#fdae6b", "#a1d99b", "#9ecae1"]


def highlight(text):
    colors, blocks = {}, []
    for sent, spans in analyse(text):
        pieces, last = [], 0
        for sp in sorted(spans, key=lambda s: s["start"]):
            if sp["start"] < last: continue
            col = colors.setdefault(sp["type"], _PALETTE[len(colors) % len(_PALETTE)])
            pieces.append(_html.escape(sent[last:sp["start"]]))
            pieces.append(
                f'<mark style="background:{col};color:#111;padding:1px 4px;border-radius:4px">'
                f'{_html.escape(sent[sp["start"]:sp["end"]])}'
                f'<sup style="font-size:.7em;opacity:.7"> {sp["type"]}</sup></mark>')
            last = sp["end"]
        pieces.append(_html.escape(sent[last:]))
        blocks.append("".join(pieces))
    return HTML("<div style='line-height:2.1;font-size:15px'>" + " ".join(blocks) + "</div>")

print("Modele charge |", device, "|", len(id2label), "labels")

Modele charge | mps | 41 labels


### Le texte long (style corpus) — espece fictive `Rutilus meridionalis`

In [2]:
LONG_TEXT = (
    "The freshwater cyprinid Rutilus meridionalis inhabits slow - flowing lowland rivers across southern Europe. "
    "At a stable water temperature of 17 ° C , the mean duration from fertilisation to hatching , i.e. , the age at birth , was recorded as 12 days . "
    "The initial wet weight of the freshly laid egg has been recorded at 0.85 grams , indicating a moderate maternal investment per offspring . "
    "Newly hatched larvae emerge possessing a total length at birth of 0.6 cm . "
    "The wet weight of newly hatched larvae is low ; upon eclosion , the wet weight at birth is approximately 1.2 grams . "
    "Under the same thermal regime , the time since birth at weaning is approximately 21 days since birth . "
    "At this point the young transition to solid foods , having reached a wet weight at weaning of 3.4 grams . "
    "Growth trajectories derived from scale annuli show that the total length at puberty is 8.2 cm . "
    "Reproductively active females exhibited a wet weight at puberty of 24 grams . "
    "Females typically attain puberty at approximately 730 days after hatching when water temperatures average 17 ° C . "
    "The time since birth at first brood for breeding females is approximately 950 days . "
    "Field surveys indicate that this species attains a maximum recorded total length of 21.5 cm . "
    "The ultimate wet weight , representing the asymptotic body mass attained by fully grown adults , has been estimated at 145 grams . "
    "A fully grown female at the ultimate size can produce approximately 320 eggs per day . "
    "At death , the gonadosomatic index ( GSI ) was recorded at 18 % at a temperature of 17 ° C . "
    "Longevity records obtained from mark - recapture studies indicate that the maximum documented life span of this species is 8 years ."
)

df = table(LONG_TEXT)
print(f"{len(df)} entites extraites  |  {df['Type'].nunique()} types de traits differents")
display(df)

18 entites extraites  |  16 types de traits differents


,Type,Texte extrait
0,T,17 ° C
1,ab,12 days
2,Ww0,0.85 grams
3,Lb,0.6 cm
4,Wwb,approximately 1.2 grams
5,tx,approximately 21 days since birth
6,Wwx,3.4 grams
7,Lp,8.2 cm
8,Wwp,24 grams
9,tp,approximately 730 days after hatching


### Le texte surligne

In [3]:
highlight(LONG_TEXT)

### Recapitulatif par type de trait

In [ ]:
counts = df["Type"].value_counts().rename_axis("Trait").reset_index(name="Occurrences")
display(counts)
print("Traits detectes :", ", ".join(sorted(df["Type"].unique())))

### A toi : ecris ton propre texte

Pour que le modele extraie bien, **reprends les formulations du tableau en haut** :
nomme explicitement le trait (« the ultimate wet weight ... is X grams », « the age at birth was recorded as X days »),
utilise les unites du corpus (**cm**, **grams**, **days**, **years**, **° C**, **%**),
et fais des **phrases courtes, un trait par phrase**.

In [ ]:
MY_TEXT = (
    "At a stable water temperature of 20 ° C , the age at birth was recorded as 9 days . "
    "The wet weight at birth is approximately 0.4 grams . "
    "This species attains a maximum recorded total length of 14.2 cm . "
    "The ultimate wet weight of fully grown adults has been estimated at 62 grams . "
    "The maximum documented life span of this species is 6 years ."
)

my_df = table(MY_TEXT)
print(f"{len(my_df)} entites extraites")
display(my_df)
display(highlight(MY_TEXT))